In [4]:
# -*- coding: utf-8 -*-
"""
WSGG模型参数优化完整实现
"""
import sys
from pathlib import Path
import time
import numpy as np
import pickle
import torch
import matplotlib.pyplot as plt
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import TensorDataset, DataLoader
from datetime import datetime
import os
import importlib
# 修复绘图时内核崩溃的关键语句
import sklearn.utils

# 项目根目录
project_root = Path.cwd().parent
sys.path.append(str(project_root))

# 导入自定义模块
from config import path
from src.utils import setup_device, get_timestamp, save_to_excel
from src.data_preparer import load_emissivity_data, prepare_tensors
from src.models.wsgg_model import WSGGModel

# 检查GPU可用性，在代码中直接指定GPU
gpu_id = 0  # 指定要使用的GPU编号，0,1,2,...等
device = torch.device(f'cuda:{gpu_id}' if torch.cuda.is_available() else 'cpu')
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 加载数据
train_filename = 'MixEmiss_p1.0_X0.2_Mr0.1-4.0_40_T40_path40.npy'
val_filename = 'MixEmiss_p1.0_X0.2_Mr0.1-4.0_15_T22_path22.npy'

train_data = load_emissivity_data(train_filename)
val_data = load_emissivity_data(val_filename)

def train_model(model, X_train, y_train, X_val, y_val, num_epochs=2000, batch_size=256, lr=0.001):
    """
    训练模型
    
    参数:
    model: WSGG模型
    Ng: 气体组分数
    X_train: 训练输入 (Mr, Tr, L)
    y_train: 训练目标 (epsilon)
    X_val: 验证输入 (Mr, Tr, L)
    y_val: 验证目标 (epsilon)
    num_epochs (int): 训练轮数
    lr (float): 学习率
    
    返回:
    model: 训练好的模型
    history: 训练历史记录
    """
    dataset = TensorDataset(X_train[0].cpu(), X_train[1].cpu(), X_train[2].cpu(), y_train.cpu())
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=True, persistent_workers=True, num_workers=8)
    
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, 'min', factor=0.5, patience=500, min_lr=1e-8)

    best_val_loss = float('inf')
    best_params = None
    history = {
        'train_loss': [],
        'val_loss': [],
        'epoch_times': [],
        'total_time': 0.0
    }
    
    # 日志文件
    timestamp = get_timestamp()
    log_file = path.RESULTS_DIR / f"coefficients_{timestamp}_Ng{model.Ng}_Exp{model.Exp}_Tref{model.T_ref}_bs{batch_size}.log"
    
    with open(log_file, 'w') as f:
        f.write("Epoch | Train Loss | Val Loss | Learning Rate\n")
        f.write("----------------------------------------\n")
        
        for epoch in range(num_epochs):
            start_time = time.time()
            model.train()
            epoch_loss = 0.0
            total_samples = 0
            
            for Mr_batch, Tr_batch, L_batch, eps_batch in dataloader:
                # 自动移动数据到设备
                Mr_batch = Mr_batch.to(device)
                Tr_batch = Tr_batch.to(device)
                L_batch = L_batch.to(device)
                eps_batch = eps_batch.to(device)
                
                optimizer.zero_grad()
                eps_pred = model(Mr_batch, Tr_batch, L_batch)
                loss = torch.sum(((eps_pred - eps_batch) / eps_batch)**2)
                # loss = torch.mean(((eps_pred - eps_batch) / eps_batch)**2)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
                # epoch_loss += loss.item() * len(eps_batch)
                # total_samples += len(eps_batch)
            
            # 平均训练损失
            # avg_train_loss = epoch_loss / total_samples
            avg_train_loss = epoch_loss / len(dataset)
            
            # 验证阶段
            model.eval()
            with torch.no_grad():
                eps_val_pred = model(X_val[0], X_val[1], X_val[2])
                val_loss = torch.mean(((eps_val_pred - y_val) / y_val)**2).item()
                # val_loss = torch.mean(torch.abs((eps_val_pred - y_val) / y_val)).item()
            
            # 记录历史
            history['train_loss'].append(avg_train_loss)
            history['val_loss'].append(val_loss)
            epoch_time = time.time() - start_time
            history['epoch_times'].append(epoch_time)
            history['total_time'] += epoch_time
            
            # 保存最佳参数
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_params = {
                    "beta": model.beta.data.clone(),
                    "gamma": model.gamma.data.clone()
                }
            
            # 学习率调整
            scheduler.step(avg_train_loss)
            
            # 日志记录
            if epoch % 500 == 0:
                current_lr = optimizer.param_groups[0]['lr']
                log_msg = (f"Epoch {epoch:04d} | Train Loss: {avg_train_loss:.4e} | "
                          f"Val Loss: {val_loss:.4e} | LR: {current_lr:.2e}")
                print(log_msg)
                f.write(log_msg + '\n')
                f.flush()
        
        # 加载最佳参数
        model.beta.data.copy_(best_params["beta"])
        model.gamma.data.copy_(best_params["gamma"])
        
        # 保存最终参数
        beta_np = model.beta.detach().cpu().numpy()
        gamma_np = model.gamma.detach().cpu().numpy()
        save_to_excel(beta_np, gamma_np, model.Ng, model.Exp, 
                  path.RESULTS_DIR / f"coefficients_{timestamp}_Ng{model.Ng}_Exp{model.Exp}_Tref{model.T_ref}_bs{batch_size}.xlsx")
        
        f.write(f"\nTotal training time: {history['total_time']:.2f} seconds\n")
    
    return model, history

Using device: cuda:0


In [ ]:
def batch_size_study(batch_sizes, Ng, Exp, Pt, X, T_ref, num_epochs=200):
    """
    批大小研究
    
    参数:
    batch_sizes (list): 要研究的批大小列表
    num_epochs (int): 每个批大小的训练轮数
    """
    results = {}
    
    for batch_size in batch_sizes:
        
        # 准备数据集
        X_train, y_train, X_val, y_val = prepare_tensors(train_data, val_data, T_ref, device)
        
        # 初始化模型
        model = WSGGModel(Ng, Exp, Pt, X, T_ref).to(device)

        print(f"\n{'='*50}")
        print(f"Starting training with batch size: {batch_size}")
        print(f"{'='*50}")

        # 打印模型概要
        print("模型可训练参数:")
        for name, param in model.named_parameters():
            print(f"{name:10} : {param.shape}")
        
        # 训练模型
        _, history = train_model(model, X_train, y_train, X_val, y_val, batch_size=batch_size, num_epochs=num_epochs, lr=0.0001)
        
        # 保存结果
        results[batch_size] = history
    
    # 保存结果
    timestamp = get_timestamp()
    results_file = path.RESULTS_DIR / f"batch_size_results_{timestamp}.pkl"
    with open(results_file, 'wb') as f:
        pickle.dump(results, f)
    
    return results

def plot_results(results):
    """绘制批大小研究结果"""
    # 创建绘图目录
    os.makedirs(path.IMG_DIR, exist_ok=True)
    timestamp = get_timestamp()
    
    plt.figure(figsize=(10, 6))
    
    # 1. 训练损失曲线比较
    plt.subplot(2, 2, 1)
    for bs, result in results.items():
        plt.semilogy(result['train_loss'], label=f'Batch Size {bs}')
    plt.xlabel('Epoch')
    plt.ylabel('Training Loss')
    plt.title('Training Loss Comparison')
    plt.legend()
    plt.grid(True, which="both", ls="-")
    
    # 2. 验证损失曲线比较
    plt.subplot(2, 2, 2)
    for bs, result in results.items():
        plt.semilogy(result['val_loss'], label=f'Batch Size {bs}')
    plt.xlabel('Epoch')
    plt.ylabel('Validation Loss')
    plt.title('Validation Loss Comparison')
    plt.legend()
    plt.grid(True, which="both", ls="-")
    
    # 3. 损失随时间变化比较
    plt.subplot(2, 2, 3)
    for bs, result in results.items():
        cumulative_time = np.cumsum(result['epoch_times'])
        plt.semilogy(cumulative_time, result['val_loss'], label=f'Batch Size {bs}')
    plt.xlabel('Time (seconds)')
    plt.ylabel('Validation Loss')
    plt.title('Validation Loss vs Time')
    plt.legend()
    plt.grid(True, which="both", ls="-")
    
    # 4. 总时间比较
    plt.subplot(2, 2, 4)
    total_times = [result['total_time'] for result in results.values()]
    plt.bar([str(bs) for bs in results.keys()], total_times)
    plt.xlabel('Batch Size')
    plt.ylabel('Total Training Time (seconds)')
    plt.title('Total Training Time Comparison')
    for i, v in enumerate(total_times):
        plt.text(i, v + 10, f"{v:.1f}s", ha='center')
    
    plt.tight_layout()
    plt.savefig(path.IMG_DIR / f"batch_size_comparison_{timestamp}.png")
    plt.show()
    
    print("Batch size comparison completed and results saved.")

# 主执行流程
if __name__ == "__main__":
    # 模型参数
    Ng = 6
    Exp = 4
    Pt = 1.0
    X = 0.2
    T_ref = 1000
    
    # 执行批大小研究
    batch_sizes = [4096, 8192, 16384, 32768]
    results = batch_size_study(batch_sizes, Ng, Exp, Pt, X, T_ref, num_epochs=800000)
    
    # 绘制结果
    plot_results(results)


Starting training with batch size: 4096
模型可训练参数:
beta       : torch.Size([7, 15])
gamma      : torch.Size([7, 5])
Epoch 0000 | Train Loss: 5.6390e-01 | Val Loss: 6.0090e-01 | LR: 1.00e-04


In [ ]:
def ng_value_study(ng_values, Exp, Pt, X, T_ref, batch_size=4096, num_epochs=200, lr=0.001):
    """
    研究不同Ng值对模型性能的影响
    
    参数:
    ng_values (list): 要研究的Ng值列表
    batch_size (int): 批量大小
    num_epochs (int): 每个Ng值的训练轮数
    lr (float): 学习率
    """
    results = {}
    
    for Ng in ng_values:
        # 准备数据集（与Ng值无关）
        X_train, y_train, X_val, y_val = prepare_tensors(train_data, val_data, T_ref, device)
        
        # 初始化模型
        model = WSGGModel(Ng, Exp, Pt, X, T_ref).to(device)

        print(f"\n{'='*50}")
        print(f"Starting training with Ng: {Ng}")
        print(f"{'='*50}")

        # 打印模型概要
        print(f"模型可训练参数:")
        for name, param in model.named_parameters():
            print(f"{name:10} : {param.shape}")
        
        # 训练模型
        _, history = train_model(model, X_train, y_train, X_val, y_val, batch_size=batch_size, num_epochs=num_epochs, lr=lr)
        
        # 保存结果
        results[Ng] = history
    
    # 保存结果
    timestamp = get_timestamp()
    results_file = path.RESULTS_DIR / f"ng_value_results_{timestamp}.pkl"
    with open(results_file, 'wb') as f:
        pickle.dump(results, f)
    
    return results

def plot_ng_results(results):
    """绘制Ng值研究结果"""
    # 创建绘图目录
    os.makedirs(path.IMG_DIR, exist_ok=True)
    timestamp = get_timestamp()
    
    plt.figure(figsize=(15, 10))
    
    # 1. 训练损失曲线比较
    plt.subplot(2, 2, 1)
    for ng, result in results.items():
        plt.semilogy(result['train_loss'], label=f'Ng = {ng}')
    plt.xlabel('Epoch')
    plt.ylabel('Training Loss')
    plt.title('Training Loss for Different Ng Values')
    plt.legend()
    plt.grid(True, which="both", ls="-")
    
    # 2. 验证损失曲线比较
    plt.subplot(2, 2, 2)
    for ng, result in results.items():
        plt.semilogy(result['val_loss'], label=f'Ng = {ng}')
    plt.xlabel('Epoch')
    plt.ylabel('Validation Loss')
    plt.title('Validation Loss for Different Ng Values')
    plt.legend()
    plt.grid(True, which="both", ls="-")
    
    # 3. 最终损失值比较
    plt.subplot(2, 2, 3)
    min_val_losses = [min(result['val_loss']) for result in results.values()]
    ng_list = list(results.keys())
    plt.plot(ng_list, min_val_losses, 'bo-')
    plt.xlabel('Ng Value')
    plt.ylabel('Minimum Validation Loss')
    plt.title('Minimum Validation Loss vs Ng Value')
    plt.grid(True, which="both", ls="-")
    
    # 标记最小损失点
    min_loss_idx = np.argmin(min_val_losses)
    plt.annotate(f'Min: {min_val_losses[min_loss_idx]:.2e}',
                 (ng_list[min_loss_idx], min_val_losses[min_loss_idx]),
                 textcoords="offset points", 
                 xytext=(0,10),
                 ha='center')
    
    # 4. 训练时间比较
    plt.subplot(2, 2, 4)
    total_times = [result['total_time'] for result in results.values()]
    plt.bar(ng_list, total_times)
    plt.xlabel('Ng Value')
    plt.ylabel('Total Training Time (seconds)')
    plt.title('Training Time Comparison')
    
    # 在柱子上添加时间标签
    for i, v in enumerate(total_times):
        plt.text(ng_list[i], v + 0.1, f"{v:.1f}s", ha='center')
    
    plt.tight_layout()
    plt.savefig(path.IMG_DIR / f"ng_value_comparison_{timestamp}.png")
    plt.show()
    
    print("Ng value comparison completed and results saved.")

# 主执行流程
if __name__ == "__main__":
    # 模型参数
    Exp = 4
    Pt = 1.0
    X = 0.2
    T_ref = 1000

    # 执行Ng值研究
    ng_values = [5, 6, 7, 8, 9, 10]  # 待研究的Ng值
    results = ng_value_study(ng_values, Exp, Pt, X, T_ref, batch_size=2048, num_epochs=100000, lr=0.0001)

    # 绘制结果
    plot_ng_results(results)

In [ ]:
# 测试pytorch gpu版本是否安装成功
import torch

if __name__ == "__main__":
    assert torch.cuda.is_available(), "CUDA不可用！"
    tensor = torch.randn(1024, 1024).cuda()
    result = tensor @ tensor.T
    print(f"矩阵乘法测试成功！结果形状: {result.shape}")
    
    print("\n完整系统信息:")
    print(f"PyTorch版本: {torch.__version__}")
    print(f"CUDA版本: {torch.version.cuda}")
    print(f"cuDNN版本: {torch.backends.cudnn.version()}")
    print(f"可用GPU数量: {torch.cuda.device_count()}")
    print(f"当前设备: {torch.cuda.current_device()}")
    print(f"设备名称: {torch.cuda.get_device_name(0)}")
    print(f"设备内存: {torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB")